In [1]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git'
REPO_DIR = Path(os.environ.get(
    'AGRIPLAN_REPO_DIR',
    '/content/IT3091---Machine-Learning-Assignment'
))
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository ready.')


Repository ready.


# Wijesinghe D.T.D. crop-yield preprocessing

This notebook is self-contained. It uses the raw production workbook, calculates yield from cultivated hectares, creates a separate 15,000+ record prepared dataset, trains a yield model, and stores the outputs without requiring the other project notebooks.

In [2]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RAW_FILE = REPO_DIR / 'Raw Datasets' / 'researchData.xlsx'
PREPROCESSED_DIR = REPO_DIR / 'Wijesinghe.D.T.D-Crop Yield Preprocessing'
OUTPUT_DIR = REPO_DIR / 'Wijesinghe.D.T.D-Crop Yield Outputs'
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
raw = pd.read_excel(RAW_FILE)
required = ['District', 'Season', 'CropCategory', 'Crop', 'Year', 'Extent', 'Production']
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

yield_data = raw[required].copy()
for column in ['Year', 'Extent', 'Production']:
    yield_data[column] = (
        yield_data[column].astype(str).str.replace(',', '', regex=False).str.strip()
    )
    yield_data[column] = pd.to_numeric(yield_data[column], errors='coerce')
for column in ['District', 'Season', 'CropCategory', 'Crop']:
    yield_data[column] = yield_data[column].astype('string').str.strip()
yield_data = yield_data.dropna(subset=required).copy()
yield_data = yield_data[(yield_data['Extent'] > 0) & (yield_data['Production'] >= 0)].copy()
yield_data['Yield_t_ha'] = yield_data['Production'] / yield_data['Extent']
yield_data = yield_data.replace([np.inf, -np.inf], np.nan).dropna(subset=['Yield_t_ha'])
yield_data = yield_data[yield_data['Yield_t_ha'] <= yield_data['Yield_t_ha'].quantile(0.995)].copy()
yield_data['Year'] = yield_data['Year'].astype(int)
yield_data = yield_data.sort_values(['Crop', 'District', 'Year', 'Season']).reset_index(drop=True)

prepared_file = PREPROCESSED_DIR / 'wijesinghe_yield_preprocessed_15000plus.csv'
yield_data.to_csv(prepared_file, index=False)
print(f'Raw source rows: {len(raw):,}')
print(f'Prepared yield rows: {len(yield_data):,}')
if len(yield_data) < 15000:
    raise ValueError('The prepared dataset is below the required 15,000 genuine records.')
print(f'Saved prepared dataset: {prepared_file.relative_to(REPO_DIR)}')
display(yield_data.head())


Raw source rows: 94,755
Prepared yield rows: 60,529
Saved prepared dataset: Wijesinghe.D.T.D-Crop Yield Preprocessing\wijesinghe_yield_preprocessed_15000plus.csv


,District,Season,CropCategory,Crop,Year,Extent,Production,Yield_t_ha
0,Ampara,Maha,Low Country Vegetable,Ash Plantain,2001,774.0,5809.0,7.505168
1,Ampara,Total,Low Country Vegetable,Ash Plantain,2001,1025.0,8087.0,7.889756
2,Ampara,Yala,Low Country Vegetable,Ash Plantain,2001,251.0,2278.0,9.075697
3,Ampara,Maha,Low Country Vegetable,Ash Plantain,2002,768.0,5780.0,7.526042
4,Ampara,Total,Low Country Vegetable,Ash Plantain,2002,790.0,5976.0,7.564557


In [3]:
features = ['District', 'Season', 'CropCategory', 'Crop', 'Year', 'Extent']
target = 'Yield_t_ha'
train_data, test_data = train_test_split(yield_data, test_size=0.2, random_state=42)
categorical = ['District', 'Season', 'CropCategory', 'Crop']
numeric = ['Year', 'Extent']
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', 'passthrough', numeric)
])
yield_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=50, max_depth=18, min_samples_leaf=3, random_state=42, n_jobs=-1))
])
yield_model.fit(train_data[features], train_data[target])
predicted = yield_model.predict(test_data[features])
metrics = pd.DataFrame([{
    'Records': len(yield_data),
    'TrainRecords': len(train_data),
    'TestRecords': len(test_data),
    'MAE_t_per_ha': mean_absolute_error(test_data[target], predicted),
    'RMSE_t_per_ha': np.sqrt(mean_squared_error(test_data[target], predicted)),
    'R2': r2_score(test_data[target], predicted)
}])
metrics.to_csv(OUTPUT_DIR / 'wijesinghe_yield_model_metrics.csv', index=False)
joblib.dump(yield_model, OUTPUT_DIR / 'wijesinghe_yield_model.joblib')

example = pd.DataFrame([{
    'District': 'Colombo',
    'Season': 'Maha',
    'CropCategory': 'Low Country Vegetable',
    'Crop': yield_data['Crop'].mode().iloc[0],
    'Year': int(yield_data['Year'].max()) + 1,
    'Extent': float(yield_data['Extent'].median())
}])
example['Predicted_Yield_t_ha'] = yield_model.predict(example[features])
example.to_csv(OUTPUT_DIR / 'wijesinghe_yield_example_prediction.csv', index=False)
display(metrics.round(4))
display(example)


,Records,TrainRecords,TestRecords,MAE_t_per_ha,RMSE_t_per_ha,R2
0,60529,48423,12106,1.9936,3.0963,0.6756


,District,Season,CropCategory,Crop,Year,Extent,Predicted_Yield_t_ha
0,Colombo,Maha,Low Country Vegetable,Chillies (Green),2026,65.0,10.225759
